# 질의 분해

두 조건을 묻는 질문을 하위 질문으로 나누어 각 근거를 찾습니다.

## 실행 환경 준비

고정한 Python 패키지를 확인합니다.

In [1]:
%pip install -q langchain==1.4.0 langchain-openai==1.6.1 python-dotenv==1.2.3 numpy==2.5.3
import sys
print("준비 완료:", sys.version.split()[0])

Note: you may need to restart the kernel to use updated packages.
준비 완료: 3.12.13


## .env 찾기

가까운 .env에서 키를 읽고 존재 여부만 표시합니다.

In [2]:
# 현재 폴더에서 가까운 .env를 찾되 비밀 값은 출력하지 않습니다.
import os, time
from dotenv import find_dotenv, load_dotenv
env_path=find_dotenv(usecwd=True)
load_dotenv(env_path)
print("OPENAI_API_KEY:","있음" if os.getenv("OPENAI_API_KEY") else "없음")

OPENAI_API_KEY: 있음


## 공통 데이터

Ch12의 축소 데이터와 Ch11 방식의 정답 문서 ID를 사용합니다.

In [3]:
# 배포 묶음에서도 같은 상대 경로를 찾도록 프로젝트 루트를 거슬러 올라갑니다.
import json
from pathlib import Path
import numpy as np
root=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/"labs").is_dir())
data_dir=root/"labs/part4/ch14-rag-patterns/data"
docs=json.loads((data_dir/"documents.json").read_text(encoding="utf-8"))
queries=json.loads((data_dir/"queries.json").read_text(encoding="utf-8"))
print("문서·질문:",len(docs),len(queries))

문서·질문: 12 8


## 모델과 임베딩

모델과 256차원 문서 벡터를 준비합니다.

In [4]:
# 256차원은 Ch12와 같은 비교 조건이며 행 하나가 문서 하나입니다.
from langchain.chat_models import init_chat_model
from langchain_openai import OpenAIEmbeddings
chat=init_chat_model("gpt-4.1-mini",model_provider="openai",temperature=0)
embedder=OpenAIEmbeddings(model="text-embedding-3-small",dimensions=256)
vectors=np.array(embedder.embed_documents([d["text"] for d in docs]))
print("임베딩 배열:",vectors.shape)

임베딩 배열: (12, 256)


## 검색·생성 함수

검색 점수, 토큰, 시간, 추정 비용을 같은 방식으로 기록합니다.

In [5]:
# 코사인 유사도는 내적을 두 벡터의 길이로 나누어 계산합니다.
from typing import Any
usage_log: list[dict[str, float | int]] = []

def search(question: str, k: int = 3) -> list[tuple[dict[str, Any], float]]:
    q=np.array(embedder.embed_query(question))
    scores=vectors@q/(np.linalg.norm(vectors,axis=1)*np.linalg.norm(q))
    # argsort 결과를 뒤집어 점수가 큰 문서부터 k개를 고릅니다.
    return [(docs[i],float(scores[i])) for i in np.argsort(scores)[::-1][:k]]
def ask(prompt: str) -> tuple[str, int, float]:
    started=time.perf_counter()
    msg=chat.invoke(prompt)
    usage=msg.usage_metadata or {}
    details=usage.get("input_token_details", {})
    input_tokens=usage.get("input_tokens", 0)
    cached_tokens=details.get("cache_read", 0)
    output_tokens=usage.get("output_tokens", 0)
    tokens=usage.get("total_tokens", 0)
    # 2026-09-10 gpt-4.1-mini 공개 단가로 추정하며 실제 청구액과 다를 수 있습니다.
    usd=((input_tokens-cached_tokens)*0.40+cached_tokens*0.10+output_tokens*1.60)/1_000_000
    usage_log.append({"input":input_tokens,"cached":cached_tokens,"output":output_tokens,"usd":usd})
    return msg.text,tokens,round((time.perf_counter()-started)*1000,1)

def show_hits(hits: list[tuple[dict[str, Any], float]]) -> list[tuple[str, float]]:
    return [(d["doc_id"],round(score,3)) for d,score in hits]

## 복합 질문 검색

두 이용 조건이 들어 있는 질문을 나누기 전에 검색합니다. 기준선 상위 문서 두 건을 기록합니다.

In [6]:
# 두 사실이 들어 있는 질문을 나누기 전에 한 번 검색합니다.
question="휴학생은 전자자료와 일반 도서를 각각 이용할 수 있나요?"
print("한 번 검색:",show_hits(search(question,k=2)))

한 번 검색: [('ACC-002', 0.81), ('ACC-004', 0.47)]


## 하위 질문 생성

복합 질문을 사실별 질문 두 개로 나눕니다. 전자자료와 일반 도서 조건이 각각 남았는지 확인합니다.

In [7]:
# 전자자료와 일반 도서 조건을 각각 묻는 두 질문으로 나눕니다.
text,tokens,latency=ask(f"다음 질문을 사실 하나씩 묻는 하위 질문 2개로 나누고 줄마다 쓰세요:{question}")
subquestions=[x.lstrip("-12. ") for x in text.splitlines() if x.strip()][:2]
print(subquestions,"토큰:",tokens,"ms:",latency)

['휴학생은 전자자료를 이용할 수 있나요?  ', '휴학생은 일반 도서를 이용할 수 있나요?'] 토큰: 75 ms: 999.0


## 각 질문을 따로 검색

각 하위 질문에서 1위 원문을 고릅니다. 중복을 제거한 근거 문서 수가 늘었는지 확인합니다.

In [8]:
# 하위 질문마다 가장 가까운 근거 문서를 하나씩 고릅니다.
evidence=[]
for sub in subquestions:
    hit=search(sub,k=1)[0]
    evidence.append(hit)
    print(sub,"->",hit[0]["doc_id"],round(hit[1],3))
print("서로 다른 근거 수:",len({d["doc_id"] for d,_ in evidence}))

휴학생은 전자자료를 이용할 수 있나요?   -> ACC-002 0.825


휴학생은 일반 도서를 이용할 수 있나요? -> ACC-002 0.692
서로 다른 근거 수: 1
